In [1]:
!pip install -q torch transformers scipy phonemizer numpy fastapi uvicorn pyngrok nest_asyncio
!sudo apt-get install -y espeak-ng
!pip install -q git+https://github.com/hexgrad/kokoro.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.8/103.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 36.0 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
The following NEW packages will be installed:
  espeak-ng espeak-ng-data libespeak-ng1 libpcaudio0 libsonic0
0 upgraded, 5 newly installed, 0 to remove and 41 not upgraded.
Need to get 4,526 kB of archives.
After this operation, 11.9 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libpcaudio0 amd64 1.1-6build2 [8,956 B]
Get:2 http://archive.ubun

In [2]:
import torch
from kokoro import KPipeline
import numpy as np
import scipy.io.wavfile
import io

print("🚀 Loading Kokoro Model...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Initialize Pipeline for Hindi (supports mixed English/Hindi)
# lang_code='h' is for Hindi.
pipeline = KPipeline(lang_code='h', device=device)

print(f"✅ Kokoro Loaded on {device}!")


🚀 Loading Kokoro Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

kokoro-v1_0.pth:   0%|          | 0.00/327M [00:00<?, ?B/s]

✅ Kokoro Loaded on cuda!


In [5]:
from fastapi import FastAPI, Response
import uvicorn
import nest_asyncio
from pyngrok import ngrok
from google.colab import userdata # Import userdata to access Colab secrets
import threading # Import threading

app = FastAPI()

@app.post("/synthesize")
async def synthesize(text: str, language: str = "h"):
    """
    Generates audio from text using Kokoro.
    """
    try:
        # Generate Audio
        # voice='hf_alpha' is a good female voice for Hindi/English mix
        # You can swap this with other voice codes like 'af_bella', etc.
        generator = pipeline(
            text,
            voice='af_bella', # Choose a voice (af_bella, af_sarah, etc.)
            speed=1.0,
            split_pattern=r'\n+'
        )

        # Kokoro returns a generator of (graphemes, phonemes, audio) tuples
        all_audio = []
        for _, _, audio in generator:
            all_audio.append(audio)

        if not all_audio:
            return Response(content="No audio generated", status_code=500)

        # Concatenate all chunks
        final_audio = np.concatenate(all_audio)

        # Convert to WAV bytes
        byte_io = io.BytesIO()
        scipy.io.wavfile.write(byte_io, 24000, final_audio)

        return Response(content=byte_io.getvalue(), media_type="audio/wav")

    except Exception as e:
        return Response(content=str(e), status_code=500)

# Start Tunnel
# Retrieve NGROK_AUTH_TOKEN from Colab secrets
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    try:
        public_url = ngrok.connect(8001).public_url
        print(f"🎙️ Kokoro Voice Factory Active at: {public_url}")
    except Exception as e:
        print(f"Error connecting ngrok: {e}. Please ensure your NGROK_AUTH_TOKEN in Colab secrets is valid and up-to-date.")
        public_url = None
else:
    print("NGROK_AUTH_TOKEN not found in Colab secrets. Please add it to your Colab secrets.")
    public_url = None


# Start Server (in a separate thread to avoid asyncio event loop issues)
if public_url: # Only start server if ngrok tunnel was successful
    nest_asyncio.apply() # Apply nest_asyncio to allow asyncio to run in already running loop
    def run_uvicorn():
        uvicorn.run(app, port=8001)

    thread = threading.Thread(target=run_uvicorn)
    thread.start()
    print("FastAPI server started in a background thread.")
else:
    print("FastAPI server not started due to ngrok connection error.")

🎙️ Kokoro Voice Factory Active at: https://unarresting-lightfootedly-misty.ngrok-free.dev
FastAPI server started in a background thread.
